In [1]:
import numpy as np
import znnl as nl

import pandas as pd

from flax import linen as nn
import optax

import matplotlib.pyplot as plt
from neural_tangents import stax

import h5py as hf
from scipy.stats import pearsonr

from rich.progress import track

import seaborn as sns

Using backend: cpu

Available hardware:

TFRT_CPU_0

## Download the data

In [2]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']

raw_dataset = pd.read_csv(url, names=column_names,
                          na_values='?', comment='\t',
                          sep=' ', skipinitialspace=True)

dataset = raw_dataset.copy()
dataset = dataset.dropna()
dataset['Origin'] = dataset['Origin'].map({1: 'USA', 2: 'Europe', 3: 'Japan'})
dataset = pd.get_dummies(dataset, columns=['Origin'], prefix='', prefix_sep='')


dataset = (dataset-dataset.mean())/dataset.std()

class MPGDataGenerator(nl.data.DataGenerator):
    """
    Data generator for the MPG dataset.
    """
    def __init__(self, dataset: pd.DataFrame):
        """
        Constructor for the data generator.
        
        Parameters
        ----------
        dataset
        """        
        train_ds = dataset.sample(frac=0.8, random_state=0)
        train_labels = train_ds.pop("MPG")
        test_ds = dataset.drop(train_ds.index)
        test_labels = test_ds.pop("MPG")
        
        self.train_ds = {"inputs": train_ds.to_numpy(), "targets": train_labels.to_numpy().reshape(-1, 1)}
        self.test_ds = {"inputs": test_ds.to_numpy(), "targets": test_labels.to_numpy().reshape(-1, 1)}
        
        self.data_pool = self.train_ds["inputs"]
        
generator = MPGDataGenerator(dataset)

In [15]:
ensembles = 10

for i in range(ensembles):
    network = stax.serial(
        stax.Dense(128, parameterization='standard', b_std=1.00),
        stax.Relu(),
        stax.Dense(128, parameterization='standard', b_std=1.00),
        stax.Relu(),
        stax.Dense(1, parameterization='standard', b_std=1.0),
    )

    optimizer = nl.optimizers.TraceOptimizer(
        scale_factor=1.0, subset=0.5, 
    )
#     optimizer = optax.adam(1e-3)

    model = nl.models.NTModel(
            nt_module=network,
            optimizer=optimizer,
            input_shape=(1, 9),
    )
    
    train_recorder = nl.training_recording.JaxRecorder(
        name=f"traceopt_train_recorder_{i}",
        loss=True,
        accuracy=False,
        update_rate=1
    )
    test_recorder = nl.training_recording.JaxRecorder(
        name=f"traceopt_test_recorder_{i}",
        loss=True,
        accuracy=False,
        update_rate=1
    )
    
    train_recorder.instantiate_recorder(
        data_set=generator.train_ds
    )
    test_recorder.instantiate_recorder(
        data_set=generator.test_ds
    )
    
    training_strategy = nl.training_strategies.SimpleTraining(
        model=model, 
        loss_fn=nl.loss_functions.MeanPowerLoss(order=2),
        recorders=[train_recorder, test_recorder],
    )
    _ = training_strategy.train_model(
            train_ds=generator.train_ds,
            test_ds=generator.test_ds, 
            epochs=50, 
            batch_size=32,
            recorders=[train_recorder, test_recorder],
        )
    
    train_recorder.dump_records()
    test_recorder.dump_records()

Epoch: 0:   0%|                                                           | 0/50 [00:00<?, ?batch/s]

28288.305


Epoch: 2:   4%|█▍                                  | 2/50 [00:01<00:36,  1.30batch/s, test_loss=nan]

nan
nan


Epoch: 4:   8%|██▉                                 | 4/50 [00:02<00:16,  2.77batch/s, test_loss=nan]

...

nan
nan


Epoch: 16:  32%|██████████▉                       | 16/50 [00:04<00:05,  6.07batch/s, test_loss=nan]

nan
nan


Epoch: 17:  34%|███████████▌                      | 17/50 [00:04<00:08,  3.95batch/s, test_loss=nan]


KeyboardInterrupt: 

## Analysis

In [ ]:
def load_data(file):
    with hf.File(file, "r") as db:
        data = db["loss"][:]
        
    return data

In [ ]:
adam_data = []

for i in range(10):
    adam_data.append(
        load_data(f"adam_test_recorder_{i}.h5")
    )

In [ ]:
traceopt_data = []

for i in range(10):
    traceopt_data.append(
        load_data(f"traceopt_test_recorder_{i}.h5")
    )

In [ ]:
x = np.linspace(1, 51, 50)

plt.errorbar(
    x,
    np.mean(traceopt_data, axis=0), 
    yerr=np.std(traceopt_data, axis=0),
    marker='x',
    c = "#140D4F",
    mfc="none",
    linestyle="none",
    label="TraceOpt"
)
plt.errorbar(
    x,
    np.mean(adam_data, axis=0), 
    yerr=np.std(adam_data, axis=0),
    marker='o',
    c="#4EA699",
    mfc="none",
    linestyle="none",
    label="ADAM"
)

plt.xlabel("Epoch")
plt.ylabel(r"$\mathcal{L}_{test}$")
plt.yscale("log")
plt.xscale("log")
plt.legend()
plt.savefig("mpg-vs-adam.pdf")
plt.show()